# Day 09. Exercise 03
# Ensembles

## 0. Imports

In [1]:
import warnings
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, BaggingClassifier, StackingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, precision_score, recall_score
from joblib import dump

warnings.filterwarnings('ignore')
pd.options.display.max_rows = 10

## 1. Preprocessing

1. Create the same dataframe as in the previous exercise.
2. Using `train_test_split` with parameters `test_size=0.2`, `random_state=21` get `X_train`, `y_train`, `X_test`, `y_test` and then get `X_train`, `y_train`, `X_valid`, `y_valid` from the previous `X_train`, `y_train`. Use the additional parameter `stratify`.

In [2]:
df = pd.read_csv('../data/day-of-week-not-scaled.csv')
X = df
enrich = pd.read_csv('../data/dayofweek.csv')
y = enrich['dayofweek'].astype('float')
X

,numTrials,hour,uid_user_0,uid_user_1,uid_user_10,uid_user_11,uid_user_12,uid_user_13,uid_user_14,uid_user_15,...,labname_lab02,labname_lab03,labname_lab03s,labname_lab05s,labname_laba04,labname_laba04s,labname_laba05,labname_laba06,labname_laba06s,labname_project1
0,1,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,3,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,4,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,5,5,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1681,9,20,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1682,6,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1683,7,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1684,8,20,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=21, stratify=y)
X_train, X_valid, y_train, y_valid = train_test_split(X_train, y_train, test_size=0.2, random_state=21, stratify=y_train)

## 2. Individual classifiers

1. Train SVM, decision tree and random forest again with the best parameters that you got from the 01 exercise with `random_state=21` for all of them.
2. Evaluate `accuracy`, `precision`, and `recall` for them on the validation set.
3. The result of each cell of the section should look like this:

```
accuracy is 0.87778
precision is 0.88162
recall is 0.87778
```

In [4]:
def calc_metrics(model, X, y) -> tuple:
    out: str = ''
    predict_col = model.predict(X)
    
    acc = accuracy_score(predict_col, y)
    precision = precision_score(y_true=y, y_pred=predict_col, average='weighted')
    recall = recall_score(y_true=y, y_pred=predict_col, average='weighted')
    
    for name, val in (('accuracy', acc), ('precision', precision), ('recall', recall)):
        out += f'{name} is {round(val, 5)}\n'
        
    return out, (round(acc, 5), round(precision,5), round(recall, 5))

In [5]:
def func(models_list: list, params_list: list) -> dict:
    out: dict = {}
    for model, params in zip(models_list, params_list):
        model = model(**params).fit(X_train, y_train)
        out[model] = calc_metrics(model, X_valid, y_valid)[0]

    return out


models = [SVC, DecisionTreeClassifier, RandomForestClassifier]
models_params = [{'random_state': 21, 'probability': True, 'C': 10, 'class_weight': None, 'gamma': 'auto', 'kernel':'rbf'}, 
                 {'random_state': 21, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 22},
                 {'random_state': 21, 'class_weight': None, 'criterion': 'gini', 'max_depth': 28, 'n_estimators':50}]

res = func(models, models_params)

for x in res:
    print(x, res[x], sep='\n')

SVC(C=10, gamma='auto', probability=True, random_state=21)
accuracy is 0.87778
precision is 0.88162
recall is 0.87778

DecisionTreeClassifier(class_weight='balanced', max_depth=22, random_state=21)
accuracy is 0.86667
precision is 0.86984
recall is 0.86667

RandomForestClassifier(max_depth=28, n_estimators=50, random_state=21)
accuracy is 0.89259
precision is 0.89361
recall is 0.89259



## 3. Voting classifiers

1. Using `VotingClassifier` and the three models that you have just trained, calculate the `accuracy`, `precision`, and `recall` on the validation set.
2. Play with the other parameteres.
3. Calculate the `accuracy`, `precision` and `recall` on the test set for the model with the best weights in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).

In [6]:
svc = SVC(random_state=21, probability=True, C=10, class_weight=None, gamma='auto', kernel='rbf')
dec_tree = DecisionTreeClassifier(random_state=21, class_weight='balanced', criterion='gini', max_depth=22)
rand_forest = RandomForestClassifier(random_state=21, class_weight=None, criterion='gini', max_depth=28, n_estimators=50)

for weights in ([1, 1, 1], [1.5, 1, 1.5], [2, 1, 2], [4, 1, 4]):
    for voting in ('soft', 'hard'):
        eclf = VotingClassifier(estimators=[('svc', svc), ('dec_tree', dec_tree), ('rand_forest', rand_forest)], 
                                voting=voting,  weights=weights).fit(X_train, y_train)
        print(eclf, calc_metrics(eclf, X_valid, y_valid)[0], sep='\n')

VotingClassifier(estimators=[('svc',
                              SVC(C=10, gamma='auto', probability=True,
                                  random_state=21)),
                             ('dec_tree',
                              DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=22,
                                                     random_state=21)),
                             ('rand_forest',
                              RandomForestClassifier(max_depth=28,
                                                     n_estimators=50,
                                                     random_state=21))],
                 voting='soft', weights=[1, 1, 1])
accuracy is 0.88148
precision is 0.88418
recall is 0.88148

VotingClassifier(estimators=[('svc',
                              SVC(C=10, gamma='auto', probability=True,
                                  random_state=21)),
                             ('dec_tree',
         

In [7]:
eclf = VotingClassifier(estimators=[('svc', svc), ('dec_tree', dec_tree), ('rand_forest', rand_forest)], 
                        voting='soft',  weights=[4, 1, 4]).fit(X_train, y_train)
print(eclf, calc_metrics(eclf, X_test, y_test)[0], sep='\n')

VotingClassifier(estimators=[('svc',
                              SVC(C=10, gamma='auto', probability=True,
                                  random_state=21)),
                             ('dec_tree',
                              DecisionTreeClassifier(class_weight='balanced',
                                                     max_depth=22,
                                                     random_state=21)),
                             ('rand_forest',
                              RandomForestClassifier(max_depth=28,
                                                     n_estimators=50,
                                                     random_state=21))],
                 voting='soft', weights=[4, 1, 4])
accuracy is 0.89645
precision is 0.9004
recall is 0.89645



## 4. Bagging classifiers

1. Using `BaggingClassifier` and `SVM` with the best parameters create an ensemble, try different values of the `n_estimators`, use `random_state=21`.
2. Play with the other parameters.
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision)

In [8]:
for x in range(10, 101, 10):
    eclf = BaggingClassifier(estimator=svc, n_estimators=x, random_state=21).fit(X_train, y_train)
    print(eclf, calc_metrics(eclf, X_valid, y_valid)[0], sep='\n')

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                                random_state=21),
                  random_state=21)
accuracy is 0.88519
precision is 0.89427
recall is 0.88519

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                                random_state=21),
                  n_estimators=20, random_state=21)
accuracy is 0.88519
precision is 0.89258
recall is 0.88519

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                                random_state=21),
                  n_estimators=30, random_state=21)
accuracy is 0.88889
precision is 0.89718
recall is 0.88889

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                                random_state=21),
                  n_estimators=40, random_state=21)
accuracy is 0.88148
precision is 0.89111
recall is 0.88148

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                   

In [9]:
eclf = BaggingClassifier(estimator=svc, n_estimators=30, random_state=21).fit(X_train, y_train)
print(eclf, calc_metrics(eclf, X_test, y_test)[0], sep='\n')

BaggingClassifier(estimator=SVC(C=10, gamma='auto', probability=True,
                                random_state=21),
                  n_estimators=30, random_state=21)
accuracy is 0.87278
precision is 0.8784
recall is 0.87278



## 5. Stacking classifiers

1. To achieve reproducibility in this case you will have to create an object of cross-validation generator: `StratifiedKFold(n_splits=n, shuffle=True, random_state=21)`, where `n` you will try to optimize (the details are below).
2. Using `StackingClassifier` and the three models that you have recently trained, calculate the `accuracy`, `precision` and `recall` on the validation set, try different values of `n_splits` `[2, 3, 4, 5, 6, 7]` in the cross-validation generator and parameter `passthrough` in the classifier itself,
3. Calculate the `accuracy`, `precision`, and `recall` for the model with the best parameters (in terms of accuracy) on the test set (if there are several of them with equal values, choose the one with the higher precision). Use `final_estimator=LogisticRegression(solver='liblinear')`.

In [10]:
best: tuple = (0, None, (0,0,0))

for n_splits in range(2, 8):
    skf = StratifiedKFold(n_splits=n_splits, random_state=21, shuffle=True)
    avg_accuracy_on_cv, avg_precision_on_cv, avg_recall_on_cv = 0, 0, 0

    for train_ix, test_ix in skf.split(X, y):
            
            y_model = y.iloc[train_ix]; y_test_cv = y.iloc[test_ix]
            X_model = X.iloc[train_ix]; X_test_cv = X.iloc[test_ix]
            X_train_cv, X_valid, y_train_cv, y_valid = train_test_split(X_model, y_model, test_size=0.2, random_state=21)

            eclf = StackingClassifier(estimators=[('svc', svc), ('dec_tree', dec_tree), ('rand_forest', rand_forest)], 
                                    final_estimator=LogisticRegression(solver='liblinear'), passthrough=True).fit(X_train_cv, y_train_cv)
            
            accuracy, precision, recall =  calc_metrics(eclf, X_valid, y_valid)[1]
            avg_accuracy_on_cv += accuracy
            avg_precision_on_cv += precision
            avg_recall_on_cv += recall

    avg_accuracy_on_cv = round(avg_accuracy_on_cv / n_splits, 5)
    avg_precision_on_cv = round(avg_precision_on_cv / n_splits, 5)
    avg_recall_on_cv = round(avg_recall_on_cv / n_splits, 5)

    print(f'n_splits={n_splits}', f'{avg_accuracy_on_cv} {avg_precision_on_cv} {avg_recall_on_cv}')

    if avg_accuracy_on_cv > best[2][0]:
          best = (n_splits, eclf, (avg_accuracy_on_cv, avg_precision_on_cv, avg_recall_on_cv))

print(f'BEST n_splits={best[0]}', best[1], f'accuracy={best[2][0]}', f'precision={best[2][1]}', f'recall={best[2][2]}')

n_splits=2 0.8432 0.84567 0.8432
n_splits=3 0.86963 0.87477 0.86963
n_splits=4 0.917 0.91955 0.917
n_splits=5 0.92 0.92169 0.92
n_splits=6 0.90392 0.9061 0.90392
n_splits=7 0.91355 0.91644 0.91355
BEST n_splits=5 StackingClassifier(estimators=[('svc',
                                SVC(C=10, gamma='auto', probability=True,
                                    random_state=21)),
                               ('dec_tree',
                                DecisionTreeClassifier(class_weight='balanced',
                                                       max_depth=22,
                                                       random_state=21)),
                               ('rand_forest',
                                RandomForestClassifier(max_depth=28,
                                                       n_estimators=50,
                                                       random_state=21))],
                   final_estimator=LogisticRegression(solver='liblinear'),
              

In [11]:
eclf = StackingClassifier(estimators=[('svc', svc), ('dec_tree', dec_tree), ('rand_forest', rand_forest)], 
                        final_estimator=LogisticRegression(solver='liblinear'), passthrough=True).fit(X_train, y_train)
print(eclf, calc_metrics(eclf, X_test, y_test)[0], sep='\n')


StackingClassifier(estimators=[('svc',
                                SVC(C=10, gamma='auto', probability=True,
                                    random_state=21)),
                               ('dec_tree',
                                DecisionTreeClassifier(class_weight='balanced',
                                                       max_depth=22,
                                                       random_state=21)),
                               ('rand_forest',
                                RandomForestClassifier(max_depth=28,
                                                       n_estimators=50,
                                                       random_state=21))],
                   final_estimator=LogisticRegression(solver='liblinear'),
                   passthrough=True)
accuracy is 0.90237
precision is 0.90434
recall is 0.90237



## 6. Predictions

1. Choose the best model in terms of accuracy (if there are several of them with equal values, choose the one with the higher precision).
2. Analyze: for which weekday your model makes the most errors (in % of the total number of samples of that class in your full dataset), for which labname and for which users.
3. Save the model.

In [12]:
eclf = StackingClassifier(estimators=[('svc', svc), ('dec_tree', dec_tree), ('rand_forest', rand_forest)], 
                        final_estimator=LogisticRegression(solver='liblinear'), passthrough=True).fit(X_train, y_train)
print(eclf, calc_metrics(eclf, X_test, y_test)[0], sep='\n')

StackingClassifier(estimators=[('svc',
                                SVC(C=10, gamma='auto', probability=True,
                                    random_state=21)),
                               ('dec_tree',
                                DecisionTreeClassifier(class_weight='balanced',
                                                       max_depth=22,
                                                       random_state=21)),
                               ('rand_forest',
                                RandomForestClassifier(max_depth=28,
                                                       n_estimators=50,
                                                       random_state=21))],
                   final_estimator=LogisticRegression(solver='liblinear'),
                   passthrough=True)
accuracy is 0.90237
precision is 0.90434
recall is 0.90237



In [13]:
predict_col = eclf.predict(X_test)
y_test_compare = y_test.reset_index(drop=True).value_counts().sort_values()
predict_compare = pd.Series(predict_col).value_counts().sort_values()
compare = pd.concat([predict_compare, y_test_compare], keys=['predict', 'as_is'], axis=1)
compare['%_error'] = abs(compare['predict'].values / compare['as_is'].values - 1 )
compare.sort_values(by='%_error', ascending=False)

,predict,as_is,%_error
0.0,21,27,0.222222
6.0,80,71,0.126761
4.0,20,21,0.047619
1.0,53,55,0.036364
2.0,29,30,0.033333
3.0,82,80,0.025000
5.0,53,54,0.018519


In [14]:
y_test_compare = y_test.to_frame()
y_test_compare['predict'] = predict_col

out: dict = {'uid': [], '%_error_count': [], 'row_num': []}
for x in X_test:
    if 'uid' in x:
         user_series = X_test[x].loc[X_test[x] == 1]
         user_y_test = y_test_compare.loc[user_series.index]
         compare = pd.concat([user_series, user_y_test], axis=1)
         compare['%_error'] = abs(compare['predict'].values / compare['dayofweek'].values - 1)
         error_count = compare['%_error'].loc[compare['%_error'] > 0].count() / compare['%_error'].count()
         if error_count > 0:
             out['uid'].append(x)
             out['%_error_count'].append(error_count)
             out['row_num'].append(compare.shape[0])

pd.DataFrame(out).sort_values(by='%_error_count', ascending=False)

,uid,%_error_count,row_num
7,uid_user_22,1.000000,1
16,uid_user_6,0.500000,4
12,uid_user_3,0.272727,14
2,uid_user_16,0.250000,5
4,uid_user_19,0.210526,19
...,...,...,...
8,uid_user_24,0.090909,11
9,uid_user_25,0.090909,22
15,uid_user_4,0.074074,27
6,uid_user_21,0.071429,14


In [15]:
out: dict = {'labname': [], '%_error_count': [], 'row_num': []}
for x in X_test:
    if 'labname' in x:
         lab_series = X_test[x].loc[X_test[x] == 1]
         lab_y_test = y_test_compare.loc[lab_series.index]
         compare = pd.concat([lab_series, lab_y_test], axis=1)
         compare['%_error'] = abs(compare['predict'].values / compare['dayofweek'].values - 1)
         error_count = compare['%_error'].loc[compare['%_error'] > 0].count() / compare['%_error'].count()
         
         if error_count > 0:
             out['labname'].append(x)
             out['%_error_count'].append(error_count)
             out['row_num'].append(compare.shape[0])

pd.DataFrame(out).sort_values(by='%_error_count', ascending=False)

,labname,%_error_count,row_num
1,labname_lab03,1.000000,1
3,labname_laba04,0.257143,35
4,labname_laba04s,0.240000,25
5,labname_laba06,0.222222,9
2,labname_lab05s,0.166667,6
6,labname_laba06s,0.133333,15
0,labname_code_rvw,0.076923,13
7,labname_project1,0.065476,186


In [16]:
with open("03_ensembles.joblib", "wb") as f:
    dump(eclf, f, protocol=-1)
    # The optional protocol argument, an integer, tells the pickler to use the given protocol; 
    # supported protocols are 0 to HIGHEST_PROTOCOL. If not specified, the default is DEFAULT_PROTOCOL. 
    # If a negative number is specified, HIGHEST_PROTOCOL is selected.